# 03 – Handling Missing Values

Real data is never perfect.

When you load a dataset from the real world — a CSV export, a form submission, a database dump — some values will always be missing. A student didn't fill in their marks. A sensor skipped a reading. Someone left a form field blank.

If you ignore missing values and run calculations on them, you'll get wrong results — or your code will crash.

Pandas represents missing values as **`NaN`** which stands for **Not a Number**.

In this notebook you'll learn:
- How to find where missing values are
- How to remove them
- How to fill them in with something sensible

This is called **data cleaning** — and it's usually the first thing you do before any analysis.

## Setting Up — Creating a DataFrame with Missing Values

We use `numpy` to insert `NaN` values. `np.nan` is the standard way to represent missing data in pandas.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Name":       ["Amit", "Priya", "Ravi", "Neha", "Karan", "Sneha"],
    "Marks":      [85, np.nan, 78, 92, np.nan, 74],
    "City":       ["Pune", "Mumbai", np.nan, "Pune", "Mumbai", np.nan],
    "Department": ["Science", "Arts", "Science", np.nan, "Arts", "Commerce"]
}

df = pd.DataFrame(data)

print(df)

You can see `NaN` showing up in the output where values are missing. This is exactly what a real-world dataset looks like when it has gaps.

## 1) Finding Missing Values

Before you fix anything, you need to know **where** the missing values are and **how many** there are.

In [ ]:
# isnull() returns True wherever a value is missing

print(df.isnull())

This gives a full True/False table — useful to see the pattern, but hard to summarize. Let's count instead.

In [ ]:
# Count missing values per column

print(df.isnull().sum())

This is the most useful check. You can immediately see:
- `Marks` has 2 missing values
- `City` has 2 missing values
- `Department` has 1 missing value
- `Name` has none

This tells you which columns need attention before you start working with the data.

In [ ]:
# Total missing values across the entire DataFrame

print("Total missing values:", df.isnull().sum().sum())

In [ ]:
# What percentage of each column is missing?
# Useful when working with large datasets

missing_percent = (df.isnull().sum() / len(df)) * 100
print(missing_percent)

In [ ]:
# Show only rows that have at least one missing value

print(df[df.isnull().any(axis=1)])

`any(axis=1)` checks across columns (left to right) for each row. If any value in that row is `NaN`, it returns `True` for that row.

## 2) Dropping Missing Values

The simplest fix — remove any row that has a missing value.

Use this when:
- The number of missing rows is small
- Losing those rows won't affect your analysis

In [ ]:
# dropna() removes every row that has even one NaN

df_clean = df.dropna()
print(df_clean)

Notice — `dropna()` does NOT change the original `df`. It returns a new DataFrame. That's why we saved it into `df_clean`.

If you want to modify `df` directly, add `inplace=True`.

In [ ]:
# Drop rows only if ALL values in that row are missing
# how="all" means: only drop if every column in the row is NaN

df_less_strict = df.dropna(how="all")
print(df_less_strict)

In [ ]:
# Drop rows that have missing values only in specific columns
# Here: only drop a row if Marks is missing

df_marks_clean = df.dropna(subset=["Marks"])
print(df_marks_clean)

`subset` is very handy — it lets you be selective. Instead of throwing away an entire row just because `City` is missing, you only drop rows where the column you actually care about is missing.

In [ ]:
# Drop columns that have any missing values
# axis=1 means operate on columns instead of rows

df_no_missing_cols = df.dropna(axis=1)
print(df_no_missing_cols)

Only `Name` survived — because it was the only column with no missing values. This is rarely useful in practice, but it's good to know.

## 3) Filling Missing Values

Dropping rows means losing data. Sometimes that's not acceptable — especially if your dataset is already small.

`fillna()` lets you replace missing values with something sensible instead of throwing the row away.

In [ ]:
# Fill ALL missing values in the entire DataFrame with 0
# (Not always a good idea — just showing the syntax)

print(df.fillna(0))

In [ ]:
# Fill missing Marks with the average of all existing marks
# This is the most common strategy for numeric data

avg_marks = df["Marks"].mean()
print("Average Marks:", avg_marks)

df["Marks"] = df["Marks"].fillna(avg_marks)
print(df)

Notice — `mean()` automatically skips `NaN` values. So the average is calculated from the 4 existing values, not divided by 6.

Filling with the mean is the safest default for numeric data — the overall statistics of the column barely change.

In [ ]:
# Fill missing City with "Unknown"
# For text columns, a placeholder like this is common

df["City"] = df["City"].fillna("Unknown")
print(df)

In [ ]:
# Fill missing Department with the most frequent value (the mode)
# mode() returns a Series, so [0] picks the first (most common) value

most_common_dept = df["Department"].mode()[0]
print("Most common department:", most_common_dept)

df["Department"] = df["Department"].fillna(most_common_dept)
print(df)

Three common strategies for filling missing values:

| Column type | Fill with |
|---|---|
| Numbers (marks, salary, age) | Mean or Median |
| Categories (city, department) | Mode (most frequent) or "Unknown" |
| Ordered data (time series) | Forward fill or backward fill (next section) |

## 4) Forward Fill and Backward Fill

These are special filling strategies useful when your data has an order — like dates or time series.

- **Forward fill (`ffill`)** — fill using the value from the row *above*
- **Backward fill (`bfill`)** — fill using the value from the row *below*

In [ ]:
# A temperature dataset with some missing readings

temp_data = {
    "Day":  ["Mon", "Tue", "Wed", "Thu", "Fri"],
    "Temp": [30, np.nan, np.nan, 34, 35]
}

temp_df = pd.DataFrame(temp_data)
print(temp_df)

In [ ]:
# Forward fill — carry the last known value forward
# Mon was 30, so Tue and Wed also become 30

print(temp_df["Temp"].ffill())

In [ ]:
# Backward fill — use the next known value to fill backwards
# Thu was 34, so Tue and Wed also become 34

print(temp_df["Temp"].bfill())

Forward fill is useful when data comes in a sequence and the last known value is still valid (like a sensor reading that stays steady). Backward fill is useful when you want to "look ahead" to what the next reading will be.

## 5) Mean vs Median — Which to Use?

When filling numeric missing values, you have two choices:

- **Mean** — the average. Gets pulled by extreme values (outliers).
- **Median** — the middle value. Not affected by outliers.

Let's see why it matters.

In [ ]:
salary_data = {
    "Employee": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "Salary":   [30000, 35000, np.nan, 32000, 500000]  # Eve is the CEO
}

salary_df = pd.DataFrame(salary_data)

print("Mean salary:  ", salary_df["Salary"].mean())
print("Median salary:", salary_df["Salary"].median())

The mean is skewed heavily by Eve's 5 lakh salary.

If you fill Carol's missing salary with the **mean**, you'd assign her ~150k — that's not realistic for a regular employee.

If you fill with the **median**, she gets ~32.5k — which is much more representative.

**Rule of thumb:**
- Data with outliers → use **median**
- Data without outliers → either works, **mean** is more common

In [ ]:
# Filling with median

median_salary = salary_df["Salary"].median()
salary_df["Salary"] = salary_df["Salary"].fillna(median_salary)
print(salary_df)

## 6) Checking Your Work

After cleaning, always verify that no missing values remain.

In [ ]:
# Confirm — no more NaN in our original df

print(df.isnull().sum())

In [ ]:
# notnull() is the opposite of isnull()
# Returns True where values ARE present

print(df.notnull().all())   # True means every value in that column is present

## 7) Putting It All Together — A Realistic Cleaning Workflow

In a real project, your missing value cleaning usually follows this sequence:

1. Load the data
2. Check how many values are missing per column
3. Decide what to do with each column
4. Apply and verify

Let's do that end to end.

In [ ]:
# Step 1 — Raw data (simulating a messy CSV)

raw = {
    "Name":   ["Amit", "Priya", "Ravi", np.nan, "Karan"],
    "Age":    [22, np.nan, 24, 21, np.nan],
    "Score":  [85, 90, np.nan, 88, 76],
    "Grade":  ["B", "A", np.nan, "B", "C"]
}

raw_df = pd.DataFrame(raw)
print(raw_df)

In [ ]:
# Step 2 — Check missing values

print(raw_df.isnull().sum())
print()
print("Missing %:")
print((raw_df.isnull().sum() / len(raw_df)) * 100)

In [ ]:
# Step 3 & 4 — Decide and apply

# Name: can't guess a missing name — drop those rows
raw_df = raw_df.dropna(subset=["Name"])

# Age: fill with median (handles outliers safely)
raw_df["Age"] = raw_df["Age"].fillna(raw_df["Age"].median())

# Score: fill with mean
raw_df["Score"] = raw_df["Score"].fillna(raw_df["Score"].mean())

# Grade: fill with the most common grade
raw_df["Grade"] = raw_df["Grade"].fillna(raw_df["Grade"].mode()[0])

# Reset the index after dropping rows
raw_df = raw_df.reset_index(drop=True)

print(raw_df)

In [ ]:
# Step 5 — Verify: no missing values left

print(raw_df.isnull().sum())

All zeros. The data is clean and ready for analysis.

## Summary

- Missing values show up as `NaN` in pandas
- **Find them:** `df.isnull().sum()` — count per column
- **Find affected rows:** `df[df.isnull().any(axis=1)]`
- **Drop rows:** `df.dropna()` — removes rows with any NaN
- **Drop selectively:** `df.dropna(subset=["col"])` — only drop if a specific column is NaN
- **Fill with a value:** `df["col"].fillna(value)`
- **Fill numeric:** use `.mean()` for normal data, `.median()` when there are outliers
- **Fill categories:** use `.mode()[0]` for most frequent, or `"Unknown"` as a placeholder
- **Fill ordered data:** `.ffill()` or `.bfill()`
- **Verify:** after cleaning, always run `df.isnull().sum()` again to confirm

Next up: **Sorting and Creating Columns** — adding new information to your data.

## Practice Questions 1

1. Create a DataFrame of 5 employees with columns `Name`, `Age`, `Salary`, `Department` — add `NaN` to at least 3 places.
2. Check how many values are missing per column.
3. Show only the rows that have at least one missing value.

## Practice Questions 2

1. Fill missing `Age` with the median age.
2. Fill missing `Salary` with the mean salary.
3. Fill missing `Department` with `"Not Assigned"`.
4. Verify the final DataFrame has no missing values.